In [ ]:
#ICC062 - Arquitetura de Computadores 2024/2
#Trabalho Prático 07: Script de Detecção de Bordas usando GPU e Numba
##Aluno: Nycksandro Lima dos Santos
#Matricula: 22351228

import time # uso para marcar o tempo
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda
from PIL import Image #uso para imagem
import math #uso para arrendondar
from google.colab import files # uso para fazer upload da imagem

In [ ]:
@cuda.jit

def convulacaoMatriz(matriz_copia, matriz_resultante, mascaraGx, mascaraGy): # i = indice linha, j = indice coluna
  i,j = cuda.grid(2) # Define i e j como índices globais que percorrem a matriz

  somatorio_convulacaoGx = 0 # Crio uma variavél acumuladora para Gx
  somatorio_convulacaoGy = 0 # Crio uma variavél acumuladora para Gy

  if(i < matriz_resultante.shape[0] and j < matriz_resultante.shape[1]):
    #Parte de Gx
    somatorio_convulacaoGx += matriz_copia[i,j] * mascaraGx[0][0]
    somatorio_convulacaoGx += matriz_copia[i,j+1] * mascaraGx[0][1]
    somatorio_convulacaoGx += matriz_copia[i,j+2] * mascaraGx[0][2]
    somatorio_convulacaoGx += matriz_copia[i+1,j] * mascaraGx[1][0]
    somatorio_convulacaoGx += matriz_copia[i+1,j+1] * mascaraGx[1][1]
    somatorio_convulacaoGx += matriz_copia[i+1,j+2] * mascaraGx[1][2]
    somatorio_convulacaoGx += matriz_copia[i+2,j] * mascaraGx[2][0]
    somatorio_convulacaoGx += matriz_copia[i+2,j+1] * mascaraGx[2][1]
    somatorio_convulacaoGx += matriz_copia[i+2,j+2] * mascaraGx[2][2]

    #Parte Gy
    somatorio_convulacaoGy += matriz_copia[i,j] * mascaraGy[0][0]
    somatorio_convulacaoGy += matriz_copia[i,j+1] * mascaraGy[0][1]
    somatorio_convulacaoGy += matriz_copia[i,j+2] * mascaraGy[0][2]
    somatorio_convulacaoGy += matriz_copia[i+1,j] * mascaraGy[1][0]
    somatorio_convulacaoGy += matriz_copia[i+1,j+1] * mascaraGy[1][1]
    somatorio_convulacaoGy += matriz_copia[i+1,j+2] * mascaraGy[1][2]
    somatorio_convulacaoGy += matriz_copia[i+2,j] * mascaraGy[2][0]
    somatorio_convulacaoGy += matriz_copia[i+2,j+1] * mascaraGy[2][1]
    somatorio_convulacaoGy += matriz_copia[i+2,j+2] * mascaraGy[2][2]

    #não usei numpy para fazer a operação de raiz quadrada pois sqrt do numpy não é compatível com numba
    matriz_resultante[i,j] = ((somatorio_convulacaoGx**2) + (somatorio_convulacaoGy**2))**0.5 # Realiza o calculo da magnitude e coloca na posição do pixel correspondente



def algoritmoSobel(matriz_entrada, tam_mascara = 3, stride_mascara = 1): # Algoritmo de Sobel que recebe uma matriz de entrada e uma matriz de saida, utilizando como tamanho da mascara = 3 e o stride da mascara = 1 por padrão
  if(tam_mascara == 3 and stride_mascara == 1): # Se a função receber os parametros certos (tamanho da máscara = 3 e stride da máscara = 1)

    #Defino as mascaras 3x3 já invertidas
    mascaraGx = np.array([[-1, 0, 1],[-2, 0, 2],[-1, 0, 1]], dtype = np.float32) # Mascara Gx já invertida
    mascaraGy = np.array([[-1,-2,-1],[0, 0, 0],[1, 2, 1]], dtype = np.float32) # Mascara Gy já invertida

    matriz_copia = np.pad(matriz_entrada,[1,1], "constant", constant_values = (0,0)).astype(dtype = np.float32) # Utilizando a função do numpy "pad" para realizar o padding da imagem

    matriz_resultante = np.zeros((len(matriz_copia[0])-2, len(matriz_copia)-2),dtype= np.float32) # Criando uma matriz cheia de zeros para servir de matriz resultante

    #Realizando o calculo para descobrir quantas mascaras serão necessária:
    largura_necessaria = math.ceil((len(matriz_copia[0]) - tam_mascara)/stride_mascara) + 1 # Aplicando a formúla para calcular quantas máscaras serão necessárias
    altura_necessaria = math.ceil((len(matriz_copia) - tam_mascara)/stride_mascara) +1 # Aplicando a formúla para calcular quantas máscaras serão necessárias

    #Calculando a quantidade de blocos necessária:
    #Se cada bloco da GPU está associado à execução de uma máscara, então o número total de blocos será o mesmo do número total de máscaras. (Na minha implementação estou usando 1 thread que aplica nas duas máscaras, soma os quadrados e tira a raiz quadrada)
    #O número total de máscaras (contando as duas como uma) = largura_necessaria * altura_necessaria
    #Logo: a grid terá a dimensão de (largura_necessaria, altura_necessaria)
    #cada bloco ficará responsável pela execução de 1 thread

    blocos_grid = (largura_necessaria, altura_necessaria) # Definindo uma tupla que receberá a dimensão da quantidade de blocos (grid)
    threads = (1,1) # 1 thread apenas

    #Copiando as variáveis necessárias para aplicação da função para a GPU e iniciando o timer de marcação (copiar a váriavel para GPU também conta para a contagem)

    inicio_tempo = time.time() #Inicio da marcação do tempo

    matriz_copia_gpu = cuda.to_device(matriz_copia) #copia para a memoria da gpu
    matriz_resultante_gpu = cuda.to_device(matriz_resultante) # copia para a memoria da gpu
    mascaraGx_gpu = cuda.to_device(mascaraGx) #copia para a memoria da gpu
    mascaraGy_gpu = cuda.to_device(mascaraGy) #copia para a memoria da gpu

    #Aplicação da função paralelizada

    convulacaoMatriz[blocos_grid, threads](matriz_copia_gpu, matriz_resultante_gpu, mascaraGx_gpu, mascaraGy_gpu) # Aplica a convolução de forma paralelizada nas duas máscaras, calculando Gx e Gy, soma os quadrados e tira a raiz, onde cada thread atua em um bloco (bloco executa as 2 máscaras)

    matriz_resultante = matriz_resultante_gpu.copy_to_host() #copia da memoria da gpu para variavel

    fim_tempo = time.time() #Fim da marcação de tempo

    tempo_decorrido = fim_tempo - inicio_tempo #Calculando quanto tempo se passou

    print(f"Tempo de execução: {tempo_decorrido} segundos")
    return matriz_resultante #retorna a matriz já com Sobel aplicada

  else: #Mensagem de erro caso esteja no formato errado
    print("O algoritmo só funciona para tamanho da mascara = 3 e stride = 1")


In [ ]:
#Célula para colocar as imagens

imagem_upload = files.upload() # Carrego uma imagem
imagem_upload = list(imagem_upload.keys())[-1] # pego o elemento inserido

imagem = Image.open(imagem_upload) # Aqui dentro coloca o caminho da imagem ou daa imagem carregada

if(imagem.mode != "L"): #Verifico se a imagem não é de escala de cinzas (2D), se for eu converto para escalas de cinza
  imagem = imagem.convert("L") # Convertendo para escala de cinza

imagem = imagem.resize((600,600)) # Utiliza o resize escolhendo a dimensão da imagem, se precisar usar, só comentar a linha

imagem_pos_filtro = algoritmoSobel(imagem) # Aplica o filtro de Sobel na imagem e obtem a matriz numpy dessa imagem

imagem_pos_filtro = Image.fromarray(imagem_pos_filtro) # Transforma a matriz numpy em uma imagem RGB

plt.axis("off") # Tirando os eixos da imagem (vou plotar a imagem)
plt.imshow(imagem_pos_filtro) # Mostrando a imagem

print(f"As dimensões das imagens são: {imagem_pos_filtro.size}") # Printo as dimensões da imagem